In [1]:
# import libraries (you may add additional imports but you may not have to)
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

In [2]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/books/book-crossings.zip

!unzip book-crossings.zip

books_filename = 'BX-Books.csv'
ratings_filename = 'BX-Book-Ratings.csv'

--2026-10-14 21:26:11--  https://cdn.freecodecamp.org/project-data/books/book-crossings.zip
Resolving cdn.freecodecamp.org (cdn.freecodecamp.org)... 104.26.3.33, 104.26.2.33, 172.67.70.149, ...
Connecting to cdn.freecodecamp.org (cdn.freecodecamp.org)|104.26.3.33|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26085508 (25M) [application/zip]
Saving to: ‘book-crossings.zip.2’

book-crossings.zip. 100%[===================>]  24.88M   118MB/s    in 0.2s    

2026-10-14 21:26:12 (118 MB/s) - ‘book-crossings.zip.2’ saved [26085508/26085508]

Archive:  book-crossings.zip
replace BX-Book-Ratings.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: BX-Book-Ratings.csv     
replace BX-Books.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: BX-Books.csv            
replace BX-Users.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: BX-Users.csv            


In [3]:
# import csv data into dataframes
df_books = pd.read_csv(
    books_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['isbn', 'title', 'author'],
    usecols=['isbn', 'title', 'author'],
    dtype={'isbn': 'str', 'title': 'str', 'author': 'str'})

df_ratings = pd.read_csv(
    ratings_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'float32'})

In [9]:
print(df_ratings.columns)
print(df_books.columns)

Index(['user', 'isbn', 'rating'], dtype='object')
Index(['isbn', 'title', 'author'], dtype='object')


In [11]:
user_counts = df_ratings['user'].value_counts()
good_users = user_counts[user_counts >= 200].index
filtered_ratings = df_ratings[df_ratings['user'].isin(good_users)]

book_counts = filtered_ratings['isbn'].value_counts()
good_books = book_counts[book_counts >= 100].index
filtered_ratings = filtered_ratings[filtered_ratings['isbn'].isin(good_books)]

df_books_with_ratings = filtered_ratings.merge(df_books, on='isbn')
book_user_pivot = df_books_with_ratings.pivot_table(
    index='title',
    columns='user',
    values='rating'
).fillna(0)

model_knn = NearestNeighbors(metric='cosine', algorithm='brute')
model_knn.fit(book_user_pivot.values)

def get_recommends(book_title=""):
    try:
        book_idx = book_user_pivot.index.get_loc(book_title)
    except KeyError:
        return f"Book '{book_title}' does not exist or was filtered out."

    distances, indices = model_knn.kneighbors(
        book_user_pivot.iloc[book_idx, :].values.reshape(1, -1),
        n_neighbors=6
    )

    recommended_books = []
    for i in range(1, 6):
        rec_title = book_user_pivot.index[indices[0][i]]
        rec_distance = float(distances[0][i])
        recommended_books.append([rec_title, rec_distance])

    return [book_title, recommended_books]

In [14]:
# function to return recommended books - this will be tested
user_counts = df_ratings['user'].value_counts()
good_users = user_counts[user_counts >= 200].index
filtered_ratings = df_ratings[df_ratings['user'].isin(good_users)]

book_counts = filtered_ratings['isbn'].value_counts()
good_books = book_counts[book_counts >= 100].index
filtered_ratings = filtered_ratings[filtered_ratings['isbn'].isin(good_books)]

df_books_with_ratings = filtered_ratings.merge(df_books, on='isbn')
book_user_pivot = df_books_with_ratings.pivot_table(
    index='title',
    columns='user',
    values='rating'
).fillna(0)

model_knn = NearestNeighbors(metric='cosine', algorithm='brute')
model_knn.fit(book_user_pivot.values)

def get_recommends(book_title = ""):
    try:
        book_idx = book_user_pivot.index.get_loc(book_title)
    except KeyError:
        return f"Book '{book_title}' does not exist or was filtered out."

    distances, indices = model_knn.kneighbors(
        book_user_pivot.iloc[book_idx, :].values.reshape(1, -1),
        n_neighbors=6
    )

    recommended_books = []
    for i in range(1, 6):
        rec_title = book_user_pivot.index[indices[0][i]]
        rec_distance = float(distances[0][i])
        recommended_books.append([rec_title, rec_distance])

    return [book_title, recommended_books]

In [15]:
books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(books)

def test_book_recommendation():
  test_pass = True
  recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
  if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
    test_pass = False
  recommended_books = ["I'll Be Seeing You", 'The Weight of Water', 'The Surgeon', 'I Know This Much Is True']
  recommended_books_dist = [0.8, 0.77, 0.77, 0.77]
  for i in range(2):
    if recommends[1][i][0] not in recommended_books:
      test_pass = False
    if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
      test_pass = False
  if test_pass:
    print("You passed the challenge! 🎉🎉🎉🎉🎉")
  else:
    print("You haven't passed yet. Keep trying!")

test_book_recommendation()

["Where the Heart Is (Oprah's Book Club (Paperback))", [['The Lovely Bones: A Novel', 0.7234864234924316], ["The Pilot's Wife : A Novel", 0.8192678093910217], ['The Joy Luck Club', 0.8198604583740234], ['The Notebook', 0.8236683011054993], ['Bel Canto: A Novel', 0.8247874975204468]]]
You haven't passed yet. Keep trying!
